Grover's search for Dynamic QMDP
- Consider the agent will start at s0 and terminate at s3. Searching '1000' starting states
Ideal Simulation

***1) Ideal Simulation***

In [ ]:
# importing packages
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister, transpile
from qiskit_aer.backends import AerSimulator
from qiskit.circuit.library import RYGate
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from typing import List

In [ ]:
from typing import List

# functions
def reverse_function(x):
    return x[::-1]

def _state_to_latex_ket(
    data: List[complex], max_size: int = 12, prefix: str = "", decimals: int = 10 
    ) :
    num = int(np.log2(len(data)))

    def ket_name(i):
        return bin(i)[2:].zfill(num)

    data = np.around(data, decimals)
    nonzero_indices = np.where(data != 0)[0].tolist()
    latex_terms = data[nonzero_indices] ### changes here 

    nextState_prop_dict= {}
    nextState_bits_dict = {}
    counter = 0
    
    for idx, ket_idx in enumerate(nonzero_indices):
            term = latex_terms[idx]
            ket = ket_name(ket_idx)
            
            # nextState_prop_dict[counter] = np.abs(np.real(term))
            nextState_prop_dict[counter] = (np.real(term))
            nextState_bits_dict[counter] = ket
            counter +=1
            
    return nextState_prop_dict,nextState_bits_dict

In [ ]:
# Classical environment

class ClassicToyEnv:

    def __init__(self, MaxSteps=np.Inf):
        
        self.MaxSteps= MaxSteps # Maximum number of steps before ending an episode
        
        # ========================================
        # MDP definition
        self.nS= 4 # Number of MDP states
        self.nA= 2 # Number of MDP actions
        self.nR= 4 # Number of possible rewards

        # # Transition function
        self.T= np.zeros((self.nS, self.nA, self.nS))
        self.T[0, 0, 1]= 0.6
        self.T[0, 0, 2]= 0.4
        self.T[0, 1, 0]= 0.1 #here
        self.T[0, 1, 1]= 0.9 #here
        self.T[1, 0, 0]= 0.2 
        self.T[1, 0, 1]= 0.8
        self.T[1, 1, 2]= 0.7
        self.T[1, 1, 3]= 0.3
        self.T[2, 0, 0]= 0.3
        self.T[2, 0, 2]= 0.7
        self.T[2, 1, 1]= 0.2
        self.T[2, 1, 3]= 0.8
        self.T[3, 0, 2]= 0.6
        self.T[3, 0, 3]= 0.4
        self.T[3, 1, 3]= 1.0
        
        # Reward function
        self.R= np.array([10, -5, 1, -10])
        
        # Current state and step
        self.currentState= None
        self.currentStep= None
        
    
    # returns the number of states in the environment
    def numberOfStates(self):
        return self.nS
       
    # returns the number of states in the environment
    def numberOfActions(self):
        return self.nA
    
    # Returns the transition probability from state s and action a to state sp
    def transitionProb(self, s, a, sp):
        return self.T[s, a, sp]
    
    # Returns the reward value for the transition (s,a)->sp, i.e. r(s,a,sp)
    def rewardValue(self, s, a, sp):
        return self.R[sp]
    
    # Resets the environment to default state
    # Returns the current env. state
    def reset(self):
        # Set default env state and step number
        self.currentState= 0
        self.currentStep= 0
        return int(self.currentState)

    def StoppingCriterionSatisfied(self):
        return self.currentStep >= self.MaxSteps

In [ ]:
# Quantum Environment for Two State Transitions with Return
#  qs --> reset + hadamard 
# Reward qubts store to Ancillary qubits before mid-circuit measurement

class QuantumToyEnv(ClassicToyEnv):
    
    # Constructor. 
    # Param MaxSteps: Maximum number of environment/agent interactions allowed for an episode
    def __init__(self, MaxSteps=np.Inf):
        super().__init__(MaxSteps)
    
        # ========================================
        # Quantum MDP representation 
        self.nqS= 2 # Number of qubits for state representation= log2(nS)
        self.nqA= 1 # Number of qubits required for action representation= log2(nA)
        self.nqR= 2 # Number of qubits for reward representation= log2(nR)
        self.nqRt = 4 # Number of qubits for return
        self.nqAnci = 2 # Number of ancillary qubits

        # Quantum Registers for MDP 2-state transition 
        self.qS= QuantumRegister(self.nqS, name='qState') # Quantum Register to store input environment states
        self.qSp= QuantumRegister(self.nqS, name='qNextState') # Quantum Register to store output environment states
        self.qA= QuantumRegister(self.nqA, name='qAction') # Quantum Register to store actions
        self.qR= QuantumRegister(self.nqR, name='qReward') # Quantum Register to store rewards
        
        self.t0_qAnci = QuantumRegister(self.nqAnci, name = 't0_qAnci') # ancillary qubits to store the timestep 0 reward
        self.t1_qAnci = QuantumRegister(self.nqAnci, name = 't1_qAnci') # ancillary qubits to store the timestep 1 reward
        
        # Quantum Registers for Return
        self.qRt= QuantumRegister(self.nqRt, name='qRt') # Quantum Register to store return
        
        # Classical Registers
        self.cReg= ClassicalRegister(25, name='cReg') # Classical Register to store measured three timesteps and return

        # Quantum Circuit to store a cycle of an MDP
        self.qc= QuantumCircuit(self.qS, self.qA, self.qSp, self.qR, self.t0_qAnci,  self.t1_qAnci, self.qRt, self.cReg) 
        
        # ----------------------------------------------------------------------------------------------------------------
        # for t = 0 , Initial step      
        # Us implementation: Appends Us to self.qc
        self.__Us_0(self.qc, self.qS)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
        # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t0_qAnci) 
        
        self.qc.barrier()
                
        """ Mid-ciruit Measurement at timestep 0 """
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[0], self.cReg[1], self.cReg[2], self.cReg[3], self.cReg[4], self.cReg[5], self.cReg[6]])
        
        self.qc.barrier()
        
        # ------------------------------------------------------------------------------------------------------------
        # for t = 1     
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 0 to t = 1. """       
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)
        
        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)
                
        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)
        
       # Uanci implementation: Store the reward at the ancillary qubits
        self.__Uanci(self.qc, self.qR, self.t1_qAnci) 
        
        self.qc.barrier()
         
        """ Mid-ciruit Measurement at timestep 1 """ 
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[7], self.cReg[8], self.cReg[9], self.cReg[10], self.cReg[11], self.cReg[12], self.cReg[13]])       
        self.qc.barrier()
                
        # -----------------------------------------------------------------------------------------------------------------
        # for t = 2
        """ Mid-ciruit Reset for 'State qubit', 'Action qubit', 'Reward qubit'. """
        self.qc.reset(self.qS)
        self.qc.reset(self.qA)
        self.qc.reset(self.qR)
        
        """ State transition from t = 1 to t = 2. """    
        # Us implementation: Appends Us to self.qc
        self.__Us(self.qc, self.qSp, self.qS)

        """ Mid-ciruit Reset for 'Next state'."""
        self.qc.reset(self.qSp)
        
        # Ua implementation: Appends Ua to self.qc
        self.__Ua(self.qc, self.qA)

        # # Ut implementation: Appends transition function to self.qc
        self.__Ut(self.qc, self.qS, self.qA, self.qSp)

        # Ur implementation: Appends Reward function to self.qc
        self.__Ur(self.qc, self.qSp, self.qR)    

        self.qc.barrier()
        
        # -----------------------------------------------------------------------------------------------------------------
        # Return
        self.__UReturn(self.qc, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt)
        
        self.qc.barrier()
      
        #  -------------------------------------------------------------------------------------------------------------------
        # # QRL (Grover's search for '1000' starting state)
        self.iterations = 1
        for i in range( self.iterations):
         """oracle (state, next state, reward at t=0, reward at t=1, reward at t=2, return)"""
         self.__UOracle(self.qc, self.qS,self.qSp, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3])
         self.qc.barrier()
         
         """diffuser  (state, action, next state, reward at t=0, reward at t=1, reward at t=2, return)"""
         self.__UDiffuser(self.qc, self.qS, self.qA, self.qSp, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3])
         self.qc.barrier()
         
        # end of iteration loop   
    
        """ Mid-ciruit Measurement at timestep 2  and Return"""
        self.qc.measure([self.qS[0], self.qS[1], self.qA[0], self.qSp[0] ,self.qSp[1], self.qR[0], self.qR[1]], 
                               [self.cReg[14], self.cReg[15], self.cReg[16], self.cReg[17], self.cReg[18], self.cReg[19], self.cReg[20]])     
        
        self.qc.measure([self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3]], 
                               [self.cReg[21], self.cReg[22], self.cReg[23], self.cReg[24]])     
        
    # Implementation of the QSample encoding
    def __QSampleNextState(self, probs, qc, qS, qA, qSp, control_s, control_a, control_sp= '', currentQ=0):
        
        all_controls= control_s + control_a + control_sp
        
        # Find control qubits
        controlsS= []
        for i in range(len(qS)):
            controlsS.append(qS[i])
        controlsA= []
        for i in range(len(qA)):
            controlsA.append(qA[i])
        controlsSp= []
        for i in range(currentQ):
            controlsSp.append(qSp[i])
        
        # Base case: End of recursion
        if currentQ>=len(qSp):
            return
        
        else: # General case
        
            # compute prob(qS[currentQ | qS, qA, qS[:currentQ-1]])
            add_prob= np.sum(probs)
            if add_prob > 0:
                current_prob= np.sum(probs[len(probs)//2:])/add_prob
                angle= 2*np.arcsin(np.sqrt(current_prob))
                
                # Make custom controlled CRY gate 
                cry= RYGate(angle).control(num_ctrl_qubits=len(all_controls), 
                                             ctrl_state= all_controls)
                
                qc.append(cry, [*controlsSp, *controlsA, *controlsS, qSp[currentQ]])
        
            self.__QSampleNextState(probs[:len(probs)//2], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'0', currentQ=currentQ+1)
            self.__QSampleNextState(probs[len(probs)//2:], qc, qS, qA, qSp, 
                    control_s= control_s,
                    control_a= control_a,
                    control_sp=control_sp+'1', currentQ=currentQ+1)
    

    # Circuit to set input environment state  
    # for state in superposition
    def __Us_0(self, qc, qS):
        for i in range(len(qS)):
            qc.h(qS[i])

    def __Us(self, qc, qSp, qS):   
        for i in range(len(qS)):
           if i == 1:
             qc.cx(qSp[i],qS[i-1])
           else:
            qc.cx(qSp[i],qS[i+1])
       
    # Circuit to set input action    
    def __Ua(self, qc, qA):
        for i in range(len(qA)):
            qc.h(qA[i])
    
    # Circuit to implement transition function
    def __Ut(self, qc, qS, qA, qSp):
        
        for s in range(self.nS): 
            for a in range(self.nA):
                
                # Conditionals for input state s
                binS= bin(s)[2:][::-1]
                while len(binS)<self.nqS:
                    binS+= '0'
                binS= binS[::-1]
                
                # Conditionals for input action a
                binA= bin(a)[2:][::-1]
                while len(binA)<self.nqA:
                    binA+= '0'
                binA= binA[::-1]
        
                # Transition Function P(s,a)
                probs= self.T[s, a]
                self.__QSampleNextState(probs, self.qc, qS, qA, qSp, 
                        control_s=binS, 
                        control_a=binA)

    # Circuit to implement reward function
    def __Ur(self, qc, qSp, qR):
        for i in range(len(qSp)):
            qc.cx(qSp[i], qR[i])

    # Circuit to store the reward at the ancillary qubits
    def __Uanci(self, qc, qR, qAnci):
        for i in range(len(qR)):
            qc.cx(qR[i], qAnci[i])       


    # Circuit to store the reward at the ancillary qubits
    def __UReturn(self, qc, t0_qReward, t1_qReward, qReward, qReturn):
      ## For Return qubit[0]
      qc.cx(t0_qReward[1], qReturn[0])
      qc.cx(t1_qReward[1], qReturn[0])
      qc.cx(qReward[1], qReturn[0])
      
      qc.barrier()

      ## For Return qubit[1]
      qc.cx(t0_qReward[0], qReturn[1])
      qc.cx(t1_qReward[0], qReturn[1])
      qc.cx(qReward[0], qReturn[1])
      # For carry-in part
      qc.ccx(t0_qReward[1], t1_qReward[1], qReturn[1])
      qc.ccx(t0_qReward[1], qReward[1], qReturn[1])
      qc.ccx(t1_qReward[1], qReward[1], qReturn[1])
      qc.barrier()
      
      # For Return qubit [2]
      # when no carry-in bit from LSB
      qc.ccx(t0_qReward[0],  t1_qReward[0], qReturn[2])
      qc.ccx(t0_qReward[0], qReward[0],qReturn[2])
      qc.ccx(t1_qReward[0], qReward[0] ,qReturn[2])

      # when carry-in bit from LSB
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      ## carry-in and single one qubit
      qc.mcx([t0_qReward[1], t1_qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], t1_qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1],  t1_qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t1_qReward[1], qReward[1],t0_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t1_qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], qReward[0]],[qReturn[2]])

      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[2]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[2]])

      qc.barrier()

      ### for  Return qubit [3] (carry-out qubit)
      qc.mcx([t0_qReward[1], t1_qReward[1], t0_qReward[0], t1_qReward[0],  qReward[0]],[qReturn[3]])
      qc.mcx([t0_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      qc.mcx([t1_qReward[1], qReward[1], t0_qReward[0], t1_qReward[0], qReward[0]],[qReturn[3]])
      
    def __UOracle(self, qc, qState, qNextState,t0_qReward,t1_qReward, t2_qReward,qReturn0, qReturn1, qReturn2, qReturn3): 
     
    # # case 18     
     qc.x([qReturn0,qReturn1,qReturn2]) 
     qc.h(qReturn3)
     qc.mcx([t0_qReward[1], t1_qReward[0],t1_qReward[1], t2_qReward[0],t2_qReward[1], qReturn0,qReturn1,qReturn2],qReturn3) # case 18
     qc.h(qReturn3)
     qc.x([qReturn0,qReturn1,qReturn2])

     
    # self.__UDiffuser(self.qc, self.qS, self.qA, self.qSp, self.t0_qAnci, self.t1_qAnci, self.qR, self.qRt[0], self.qRt[1], self.qRt[2], self.qRt[3])
    def __UDiffuser(self, qc, qState, qAction, qNextState,t0_qReward,t1_qReward, t2_qReward,qReturn0, qReturn1, qReturn2, qReturn3):

     qc.x([qState[0], qState[1]])
     qc.h([qState[0], qState[1], qAction[0]])
     qc.x([qState[0], qState[1], qAction[0]])
     qc.barrier()
     qc.h(t0_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t0_qReward[1])
     qc.h(t0_qReward[1])
     qc.barrier()
     qc.h(t1_qReward[0])
     qc.h(t1_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t1_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t1_qReward[0])
     qc.h(t1_qReward[0])
     qc.h(t1_qReward[1])
     qc.barrier()
     qc.h(t2_qReward[0])
     qc.h(t2_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t2_qReward[1])
     qc.mcx([qState[0], qState[1], qAction[0]],t2_qReward[0])
     qc.h(t2_qReward[0])
     qc.h(t2_qReward[1])
     qc.barrier()
     qc.x([qState[0], qState[1], qAction[0]])
     qc.h([qState[0], qState[1], qAction[0]])
     qc.barrier()
     
     qc.h([qReturn0, qReturn1, qReturn2, qReturn3])
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn3)
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn2)
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn1)
     qc.mcx([t0_qReward[1],t1_qReward[0],t1_qReward[1],t2_qReward[0],t2_qReward[1]],qReturn0)
     qc.h([qReturn0, qReturn1, qReturn2, qReturn3])
    

In [ ]:
# Instantiate environment
dynQMDP_gro_1000= QuantumToyEnv()
dynQMDP_gro_1000.qc.draw('mpl', fold=-1, initial_state=True) # 'fold=-1' ensures no line breaks

In [ ]:
# AER simulator (Ideal simulation)

simulator = AerSimulator(method = 'statevector')
transpile_dynQMD_gro_1000 = transpile(dynQMDP_gro_1000.qc, simulator)
results_dynQMD_gro_1000 = simulator.run(transpile_dynQMD_gro_1000, shots=1024, memory=True).result()
counts_dynQMD_gro_1000= results_dynQMD_gro_1000.get_counts()
print(len(counts_dynQMD_gro_1000))
print(counts_dynQMD_gro_1000)

In [ ]:
print ('counts_dynQMD_gro_1000:',counts_dynQMD_gro_1000) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         # return rt+ " " + t2_msb+ " "+ t1_msb + " " + t0_msb
         return rt+ t2_msb+ t1_msb + t0_msb
     return value

# Create a new dictionary with modified values
modified_counts_dynQMD_gro_1000 = {correct_qubit_order(key): value for key, value in counts_dynQMD_gro_1000.items()}
print('modified_counts_dynQMD_gro_1000:',modified_counts_dynQMD_gro_1000) 

In [ ]:
len(modified_counts_dynQMD_gro_1000)

In [ ]:
modified_counts_dynQMD_gro_1000_sorted = dict(sorted(modified_counts_dynQMD_gro_1000.items(), key= lambda item: item[0], reverse= False))
modified_counts_dynQMD_gro_1000_sorted

In [ ]:
# Get list of values from the dictionary
values = list(modified_counts_dynQMD_gro_1000_sorted.values())

# Calculate the total sum manually
total = 0
for value in values:
    total += value

# Print the result
print("Total sum:", total)

In [ ]:
modified_counts_dynQMD_gro_1000_sorted.keys()

Check with reference trajectories

In [ ]:
import os
import pickle
from qiskit.visualization import plot_histogram

file_name = './previous_static_QMDP_data/Classical Counts for Two State Transitions with Return.pkl'

if os.path.exists(file_name):
 print("...found existing Classical Measurement! Reading it...")
 with open(file_name, "rb") as file:
       counts_mdp = pickle.load(file)    

print(counts_mdp)


In [ ]:
print ('counts_mdp:',counts_mdp) 

# Function to reverse the four most significant bits
def reverse_qubit_order_msb(value):
    if len(value) >= 4:
        most_significant_four = value[:4]
        rest = value[4:]
        reversed_most_significant_four = most_significant_four[::-1]
        return reversed_most_significant_four + rest
    return value

def correct_qubit_order(value):
     if len(value) >= 21:
         # return
         rt = value[:4] 
         # for t2
         t2 = value[4:11]
         t2_msb = reverse_qubit_order_msb(t2)
         # for t1
         t1 = value[11:18]
         t1_msb = reverse_qubit_order_msb(t1)
         # for t0
         t0 = value[18:]
         t0_msb = reverse_qubit_order_msb(t0)
         return rt+t2_msb+ t1_msb+ t0_msb
     return value

# Create a new dictionary with modified values
modified_counts_mdp = {correct_qubit_order(key): value for key, value in counts_mdp.items()}
print('modified_counts_mdp:',modified_counts_mdp) 

In [ ]:
modified_counts_mdp_sorted = dict(sorted(modified_counts_mdp.items(), key= lambda item: item[0], reverse= False))
modified_counts_mdp_sorted

reference_170_trajectories_with_return = list(modified_counts_mdp_sorted.keys())
len(reference_170_trajectories_with_return)

In [ ]:
filtered_modified_counts_dynQMD_gro_1000= {key: value for key, value in modified_counts_dynQMD_gro_1000.items() if key in reference_170_trajectories_with_return}
print('filtered_classical_measurement:',len(filtered_modified_counts_dynQMD_gro_1000))

filtered_modified_counts_dynQMD_gro_1000_sorted = dict(sorted(filtered_modified_counts_dynQMD_gro_1000.items(), key= lambda item: item[1], reverse= False))
filtered_modified_counts_dynQMD_gro_1000_sorted

In [ ]:
# Take the top 10 values from the sorted list
sorted_items = sorted(filtered_modified_counts_dynQMD_gro_1000_sorted.items(), key=lambda item: item[1], reverse=True)
top_10_values = dict(sorted_items[:10])
print ('Top 10 values (1 iteration): ')
top_10_values

In [ ]:
# Take the least 5 values from the sorted list

sorted_items = sorted(filtered_modified_counts_dynQMD_gro_1000_sorted.items(), key=lambda item: item[1], reverse=False)
least_5_values =dict(sorted_items[:5]) 
least_5_values

Plotting

In [ ]:

# Prepare data for the bar chart
qStates = reference_170_trajectories_with_return # Next state qubits and reward qubits are in correct order. Stored by qSamples.
count_gro= [filtered_modified_counts_dynQMD_gro_1000.get(key, 0) for key in qStates]  # Get corresponding values from 'counts_grover', fill with 0 if missing

# Create a DataFrame 
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count_gro': count_gro
})

df.index = range(1, len(df) + 1) # setting data frame index starting from 1 not 0

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count_gro').index  
top_10_values = df.loc[top_10_indices]
print('Top_10_values:',top_10_values)

# Positions in the bit strings for s0 and s3(0-indexed)
qubit_positions_00 = [23, 24]
qubit_positions_11 = [0, 4, 5]

# Filter function to check if the specified qubits are '00' and '11'
def filter_qubits(data, positions_s0,position_s3):
    qsample_return = {}
    for key, value in data.items():
        # Check if all specified positions have the value '00' and '11'
        if all(value[pos] == '0' for pos in positions_s0):
         if all(value[pos] == '1' for pos in position_s3):
            qsample_return[key] = value
    return qsample_return

# Apply the filter for trajectories start with s0 and terminate at s3
qsample_s0_s3 = filter_qubits(dict(top_10_values['Quantum States']), qubit_positions_00, qubit_positions_11)
print('qsample_s0_s3:',qsample_s0_s3)

top_colors  = ['#ff0000','#ff0000']
color_map = dict(zip(qsample_s0_s3.keys(), top_colors))
df['Color'] = df.index.map(color_map).fillna('#0072B2')

# make a plot
fig, ax = plt.subplots(figsize=(20, 7))

# Define the width of the bars 
bar_width = 0.8

# Create the bar chart
bars = ax.bar(df['Quantum States'], df['Count_gro'], color=df['Color'], width=bar_width)

# Set the labels and title
ax.set_xlabel('Quantum Trajectory', fontsize = 24, fontfamily ='Arial', labelpad= 12)
ax.set_ylabel('Total Count', fontsize = 24, fontfamily ='Arial', labelpad= 12)
ax.set_title('Quantum Trajectories Starting at s0 and Terminating at s3, Achieving Maximum Return in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps',  
                fontsize = 18, fontfamily ='Arial')

# Rotate x-axis labels for better readability
plt.xticks(rotation=90, fontsize= 8)
plt.yticks(rotation=0, fontsize =16, fontfamily = 'Arial')

# Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.0f}', ha='center', va='bottom', fontsize = 'x-large', rotation = 0, fontfamily ='Arial')

legend_lines = [Line2D([0], [0], color=color, lw=4) for color in top_colors]
top = qsample_s0_s3
# legend_labels = [value for key, value in top.items() ]
legend_labels = [f'T-{key} "{value}"' for key, value in top.items()]
ax.legend(legend_lines, legend_labels, title='Quantum trajectories \nstarting at s0 and terminating at s3, \nachieving maximum return\n',  loc='upper left', prop={'family': 'Arial','size': 20})
leg = ax.get_legend()
plt.setp(leg.get_title(), family='Arial',fontsize = 20)

# Add space on the left and right
left_padding = -2.2  # Add more space to the left
right_padding = len(df['Quantum States']) - 0.5 + 2  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Adjust Y-Axis Limit to Add Padding
y_max = max(df['Count_gro'])  # Maximum value in the dataset
ax.set_ylim(0, y_max + 1)  # Add some padding above the tallest bar

# Show the plot
plt.tight_layout()

plt.show()

In [ ]:
from matplotlib.lines import Line2D

# Prepare data for the bar chart
qStates = reference_170_trajectories_with_return # Next state qubits and reward qubits are in correct order. Stored by qSamples.
count_gro= [filtered_modified_counts_dynQMD_gro_1000.get(key, 0) for key in qStates]  # Get corresponding values from 'counts_grover', fill with 0 if missing

# Create a DataFrame
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count_gro': count_gro
})

# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

df.index = range(1, len(df) + 1) # setting data frame index starting from 1 not 0

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count_gro').index
top_10_values = df.loc[top_10_indices]
# print('top_10_values:\n', top_10_values)

# Positions in the bit strings for s0 and s3(0-indexed)
qubit_positions_00 = [23, 24]
qubit_positions_11 = [0, 4, 5]

# Filter function to check if the specified qubits are '00' and '11'
def filter_qubits(data, positions_s0,position_s3):
    qsample_return = {}
    for key, value in data.items():
        # Check if all specified positions have the value '00' and '11'
        if all(value[pos] == '0' for pos in positions_s0):
         if all(value[pos] == '1' for pos in position_s3):
            qsample_return[key] = value
    return qsample_return

# Apply the filter for trajectories start with s0 and terminate at s3
qsample_s0_s3 = filter_qubits(dict(top_10_values['Quantum States']), qubit_positions_00, qubit_positions_11)
print('qsample_s0_s3:',qsample_s0_s3)

# Define top colors
top_colors = ['#ff0000', '#ff0000', '#ff0000', '#ff0000', '#ff0000']

# Map colors to the top 5 indices
color_map = dict(zip(qsample_s0_s3.keys(), top_colors))
df['Color'] = df.index.map(color_map).fillna('#0072B2')

# Make a plot
fig, ax = plt.subplots(figsize=(20, 6))

# Define the width of the bars
bar_width = 0.8

# Create the bar chart
bars = ax.bar(df['Index'], df['Count_gro'], color=df['Color'], width=bar_width)

# Set the labels and title
ax.set_xlabel('Quantum Trajectory', fontsize=24, fontfamily='Arial', labelpad=12)
ax.set_ylabel('Total Count', fontsize=24, fontfamily='Arial', labelpad=12)
ax.set_title('Quantum Trajectories Starting at s0 and Terminating at s3, Achieving Maximum Return in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps',  
                fontsize = 18, fontfamily ='Arial')

# Set the x-axis ticks with T- format
t_plus_labels = [f'T-{i}' for i in df['Index']]
plt.xticks(df['Index'], t_plus_labels, rotation=90, fontsize=9, fontfamily='Arial')
plt.yticks(rotation=0, fontsize=16, fontfamily='Arial')

# Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height , f'{height:.0f}', ha='center', va='bottom', fontsize='x-large', rotation=0, fontfamily='Arial')

# Create a custom legend
legend_lines = [Line2D([0], [0], color=color, lw=4) for color in top_colors]
top = qsample_s0_s3
# legend_labels = [value for key, value in top.items() ]
legend_labels = [f'T-{key} "{value}"' for key, value in top.items()]
ax.legend(legend_lines, legend_labels, title='Quantum trajectories \nstarting at s0 and terminating at s3, \nachieving maximum return\n',  loc='upper left', prop={'family': 'Arial','size': 20})
leg = ax.get_legend()
plt.setp(leg.get_title(), family='Arial',fontsize = 20)

# Add space on the left and right
left_padding = -0.5  # Add more space to the left
right_padding = len(df['Index']) - 0.5 + 2  # Add more space to the right
ax.set_xlim(left_padding, right_padding)

# Show the plot
plt.tight_layout()

plt.show()


In [ ]:
# Prepare data for the bar chart
qStates = reference_170_trajectories_with_return 
count_gro= [filtered_modified_counts_dynQMD_gro_1000.get(key, 0) for key in qStates] 

# Create a DataFrame 
df = pd.DataFrame({
    'Quantum States': qStates,
    'Count_gro': count_gro
})

df.index = range(1, len(df) + 1) # setting data frame index starting from 1 not 0

# Identify the top 10 values and their indices
top_10_indices = df.nlargest(10, 'Count_gro').index  
top_10_values = df.loc[top_10_indices]
# print('top_10_values:',top_10_values)

# Positions in the bit strings for s0 and s3(0-indexed)
qubit_positions_00 = [23, 24]
qubit_positions_11 = [0,4, 5]

# Filter function to check if the specified qubits are '00' and '11'
def filter_qubits(data, positions_s0,position_s3):
    qsample_return = {}
    for key, value in data.items():
        # Check if all specified positions have the value '00' and '11'
        if all(value[pos] == '0' for pos in positions_s0):
         if all(value[pos] == '1' for pos in position_s3):
            qsample_return[key] = value
    return qsample_return

# Apply the filter for trajectories start with s0 and terminate at s3
qsample_s0_s3 = filter_qubits(dict(top_10_values['Quantum States']), qubit_positions_00,qubit_positions_11)
print('qsample_s0_s3:',qsample_s0_s3)

top_colors  = ['#ff0000','#ff0000']
color_map = dict(zip(qsample_s0_s3.keys(), top_colors))
df['Color'] = df.index.map(color_map).fillna('#0072B2')

# Add an index column for sequential numbering
df['Index'] = range(1, len(df) + 1)

# Filter out rows with 'Count_gro' equal to 0
df_filtered = df[df['Count_gro'] > 0]

# make a plot
fig, ax = plt.subplots(figsize=(20, 6.5))

# Define the width of the bars 
bar_width = 0.7

# Create the bar chart
bars = ax.bar(df_filtered['Quantum States'], df_filtered['Count_gro'], color=df_filtered['Color'], width=bar_width)

# Set the labels and title
ax.set_xlabel('Quantum Trajectory', fontsize = 26, fontfamily ='Arial', labelpad= 12)
ax.set_ylabel('Total Count', fontsize = 26, fontfamily ='Arial', labelpad= 12)
ax.set_title('Quantum Trajectories Starting at s0 and Terminating at s3, Achieving Maximum Return in a Qubit-Reuse Quantum Markov Decision Process over Three Timesteps',  
                fontsize = 18, fontfamily ='Arial')

# Set the x-axis ticks with T+ format
t_plus_labels = [f'T-{i}' for i in df_filtered['Index']]
plt.xticks(df_filtered['Quantum States'], t_plus_labels, rotation=90, fontsize=18, fontfamily='Arial')

# # Set the y-axis ticks with intervals of 5
ax.set_yticks(range(0, int(df_filtered['Count_gro'].max()) + 1, 3))  # Tick range: 0 to max, step = 3
plt.yticks(rotation=0, fontsize=20, fontfamily='Arial')

# Add value labels on top of each bar
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height  - 0.05, f'{height:.0f}', ha='center', va='bottom', fontsize = 'xx-large', rotation = 0, fontfamily ='Arial')

top = qsample_s0_s3
legend_lines = [Line2D([0], [0], color=color, lw=4) for color in top_colors]
# legend_labels = [f'T-{key} "{value}"' for key, value in top.items()] # double quoting
legend_labels = [f"T-{key} '{value}'" for key, value in top.items()] # single quoting
ax.legend(legend_lines, legend_labels, title='Quantum trajectories \nstarting at s0 and terminating at s3, \nachieving maximum return\n',  loc='upper left', prop={'family': 'Arial','size': 22})
leg = ax.get_legend()
plt.setp(leg.get_title(), family='Arial',fontsize = 22)

# Add space on the left and right
left_padding = -0.8  # Reduce padding on the left
right_padding = len(df_filtered['Quantum States']) - 0.5 + 0.5  # Adjust padding on the right
ax.set_xlim(left_padding, right_padding)  # Apply to x-axis limits

# # Adjust Y-Axis Limit to Add Padding
y_max = max(df['Count_gro'])  # Maximum value in the dataset
ax.set_ylim(0, y_max + 1)  # Add some padding above the tallest bar

# Show the plot
plt.tight_layout()

plt.show()